<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS15/12_Deployment_Guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS15: Panduan Deployment Model Machine Learning - AntiBurnOut Web App

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H
- **Dataset**: `mental_health_burnout_prediction_dataset.csv`

---

### Deskripsi Praktikum Deployment:
Praktikum ini menyajikan alur lengkap produksi dan penerapan (*deployment*) model Machine Learning ke lingkungan cloud siap pakai (*production-ready*).

Berbeda dengan jobsheet reguler yang berfokus pada upload gambar, pada proyek **AntiBurnOut** ini sistem dideploy sebagai **Sistem Skrining Kesehatan Mental Interaktif** berbasis formulir web modern dan REST API.

### Alur Deployment Terstruktur:
1. **Pelatihan & Serialisasi Model:** Model Neural Network terbaik dilatih dan disimpan ke format `burnout_model.h5`, bersama artefak preprocessor `preprocessor.pkl`.
2. **Pembangunan Web Server (Flask):** Menyediakan rute UI (`/`) untuk antarmuka pengguna interaktif dan rute API (`/predict`) untuk inferensi programatik (format JSON).
3. **Kontainerisasi Docker:** Mengemas aplikasi, dependensi, dan artefak model ke dalam container Docker standar port `7860` (kompatibel penuh dengan **Hugging Face Spaces** dan **Railway/Render**).
4. **Pengujian Lokal & Cloud:** Verifikasi inferensi menggunakan antarmuka web browser dan perintah `curl`.

In [ ]:
# 1. Pelatihan Model Produksi & Ekspor Artefak Serialisasi
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler

# Pastikan direktori deploy tersedia
deploy_dir = 'antiburnout-deployment'
os.makedirs(deploy_dir, exist_ok=True)

# Muat dataset
possible_paths = [
    '../mental_health_burnout_prediction_dataset.csv',
    'mental_health_burnout_prediction_dataset.csv',
    '../../mental_health_burnout_prediction_dataset.csv'
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
df = pd.read_csv(csv_path)

# 10 Fitur Utama Skrining Burnout
feature_cols = [
    'Age', 'Work_Hours_Per_Week', 'Sleep_Hours', 'Work_Life_Balance',
    'Job_Satisfaction', 'Anxiety_Score', 'Depression_Score',
    'Mood_Score', 'Physical_Activity_Hours', 'Screen_Time_Hours'
]

data = df[feature_cols + ['Burnout_Risk']].copy()
for col in feature_cols:
    data[col] = data[col].fillna(data[col].median())
data['Burnout_Risk'] = data['Burnout_Risk'].fillna(data['Burnout_Risk'].mode()[0])

risk_map = {'Low': 0, 'Moderate': 1, 'High': 2}
y = data['Burnout_Risk'].map(risk_map).values
X = data[feature_cols].values

# Fit Scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Bangun & Latih Model Keras Ringan untuk Produksi
prod_model = keras.Sequential([
    keras.layers.Input(shape=(len(feature_cols),)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(3, activation='softmax')
])
prod_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
prod_model.fit(X_scaled, y, epochs=5, batch_size=128, verbose=1)

# Simpan Model ke format .h5
model_h5_path = os.path.join(deploy_dir, 'burnout_model.h5')
scaler_pkl_path = os.path.join(deploy_dir, 'preprocessor.pkl')

prod_model.save(model_h5_path)
with open(scaler_pkl_path, 'wb') as f:
    pickle.dump({'scaler': scaler, 'features': feature_cols, 'mapping': {0: 'Low', 1: 'Moderate', 2: 'High'}}, f)

print(f"Artefak Berhasil Diekspor:")
print(f"- Model Saved     : {model_h5_path} ({os.path.getsize(model_h5_path)/1024:.1f} KB)")
print(f"- Preprocessor Pkl: {scaler_pkl_path} ({os.path.getsize(scaler_pkl_path)/1024:.1f} KB)")

### Struktur Berkas Folder Deployment (`antiburnout-deployment/`)
Folder `antiburnout-deployment/` berisi seluruh komponen aplikasi web yang berdiri sendiri:

```text
antiburnout-deployment/
├── app.py                # Server web Flask (UI form & REST API /predict)
├── burnout_model.h5      # Model Neural Network yang telah dilatih
├── preprocessor.pkl      # Scaler standardisasi dan metadata kolom
├── Dockerfile            # Blueprint kontainerisasi cloud (port 7860)
├── requirements.txt      # Dependensi Python cloud
├── README.md             # Petunjuk deploy ke Hugging Face Spaces
└── templates/
    └── index.html        # Antarmuka web responsif interaktif modern
```

In [ ]:
# 2. Uji Inferensi Langsung Simulasi Pengguna
# Contoh input profil pekerja berisiko burnout tinggi:
# Usia: 28, Jam Kerja: 65, Tidur: 4.5, WLB: 2, Kepuasan: 3, Cemas: 85, Depresi: 80, Mood: 20, Olahraga: 1, Screen Time: 12
sample_input = np.array([[28, 65.0, 4.5, 2.0, 3.0, 85.0, 80.0, 20.0, 1.0, 12.0]])
sample_scaled = scaler.transform(sample_input)
prediction_probs = prod_model.predict(sample_scaled)[0]
predicted_class_idx = np.argmax(prediction_probs)
classes = ['Low', 'Moderate', 'High']

print("=== Simulasi Prediksi Web App AntiBurnOut ===")
for i, cls_name in enumerate(classes):
    print(f"- Probabilitas Risiko {cls_name:8s}: {prediction_probs[i]*100:.2f}%")
print(f"\nHasil Diagnosa: Risiko {classes[predicted_class_idx].upper()} (Keyakinan: {prediction_probs[predicted_class_idx]*100:.1f}%)")

### Langkah Deploy ke Hugging Face Spaces:
1. Buat akun di [Hugging Face](https://huggingface.co/).
2. Klik **New Space** $\rightarrow$ Berikan nama (misal: `antiburnout-screening-app`).
3. Pilih **Space SDK**: **Docker** (Blank).
4. Clone repository Space Anda ke lokal:
   ```bash
   git clone https://huggingface.co/spaces/<username>/antiburnout-screening-app
   ```
5. Salin seluruh isi folder `antiburnout-deployment/` ke repositori space tersebut.
6. Lakukan commit dan push:
   ```bash
   git add .
   git commit -m "Deploy AntiBurnOut Web App"
   git push
   ```
7. Hugging Face akan secara otomatis membangun image Docker dan web app Anda aktif secara publik di port 7860!